In [1]:
import random


class Node:
    def __init__(self, room, parent, action, step, g, h):
        self.room = room
        self.parent = parent
        self.action = action
        self.step = step
        # g(n): số bước đã đi
        self.g = g
        # h(n): số ô bẩn còn lại
        self.h = h

        # f(n) = g(n) + h(n)
        self.cost = g + h


m = int(input("Nhập số hàng: "))
n = int(input("Nhập số cột: "))

room = []

print("Nhập ma trận:")

for i in range(m):
    row = list(map(int, input().split()))
    room.append(row)

start_x = random.randint(0, m - 1)
start_y = random.randint(0, n - 1)


def room_to_tuple(room):
    return tuple(tuple(row) for row in room)


def is_goal(room):
    for row in room:
        if 1 in row:
            return False

    return True


def print_room(room, x, y):
    for i in range(m):
        for j in range(n):
            if i == x and j == y:
                print("x", end=" ")
            else:
                print(room[i][j], end=" ")

        print()

    print()


def get_rules(x, y):
    moves = []

    if x > 0:
        moves.append("UP")

    if x < m - 1:
        moves.append("DOWN")

    if y > 0:
        moves.append("LEFT")

    if y < n - 1:
        moves.append("RIGHT")

    return moves


def move(room, x, y, action):
    new_room = [row[:] for row in room]

    new_x, new_y = x, y

    if action == "UP":
        new_x -= 1

    elif action == "DOWN":
        new_x += 1

    elif action == "LEFT":
        new_y -= 1

    elif action == "RIGHT":
        new_y += 1

    # hút bụi nếu ô mới có bụi
    if new_room[new_x][new_y] == 1:
        new_room[new_x][new_y] = 0

    return new_room, new_x, new_y


# h(n): số ô bẩn còn lại
def heuristic(room):
    cnt = 0

    for i in range(m):
        for j in range(n):
            if room[i][j] == 1:
                cnt += 1

    return cnt


def print_solution(node, start_x, start_y):
    path = []

    while node is not None:
        path.append(node)
        node = node.parent

    path.reverse()
    x, y = start_x, start_y
    print("Các bước làm sạch phòng:\n")
    for node in path:
        if node.action:
            _, x, y = move(node.parent.room, x, y, node.action)
            print(f"Bước {node.step}: {node.action} | g(n) = {node.g} | h(n) = {node.h} | f(n) = {node.cost}")

        else:
            print(f"Bước 0: Vị trí bắt đầu | g(n) = {node.g} | h(n) = {node.h} | f(n) = {node.cost}")

        print_room(node.room, x, y)

    print("Tổng số bước:", len(path) - 1)
    print("Cost cuối cùng:", path[-1].cost)

def get_min_threshold(current_node, x, y, threshold):
    min_threshold = float("inf")

    for act in get_rules(x, y):
        new_room, new_x, new_y = move(current_node.room, x, y, act)

        new_g = current_node.g + 1
        new_h = heuristic(new_room)
        new_cost = new_g + new_h
        if new_cost > threshold:
            if new_cost < min_threshold:
                min_threshold = new_cost

    return min_threshold

def search_with_threshold(start_room, start_x, start_y, threshold):

    frontier = []
    reached = {}

    start_g = 0
    start_h = heuristic(start_room)

    start_node = Node(start_room, None, None, 0, start_g, start_h)

    frontier.append((start_node, start_x, start_y))

    min_threshold = float("inf")

    while frontier:
        current_node, x, y = frontier.pop()
        current_state = (room_to_tuple(current_node.room), x, y)

        if is_goal(current_node.room):
            return True, current_node

        reached[current_state] = current_node.cost
        #tìm cost để tăng lên khi không tìm được goal
        min_threshold_node = get_min_threshold(current_node, x, y, threshold)
        min_threshold = min(min_threshold, min_threshold_node)
        
        for act in get_rules(x, y):
            
            new_room, new_x, new_y = move(current_node.room, x, y, act)
            new_g = current_node.g + 1
            new_h = heuristic(new_room)
            new_node = Node(new_room, current_node, act, current_node.step + 1, new_g, new_h)

            if new_node.cost > threshold:
                continue

            new_state = (room_to_tuple(new_room), new_x, new_y)

            skip = False

            if new_state in reached and reached[new_state] <= new_node.cost:
                skip = True

            for node, fx, fy in frontier:
                state = (room_to_tuple(node.room), fx, fy)

                if state == new_state and node.cost <= new_node.cost:
                    skip = True
                    break

            if not skip:
                frontier.append((new_node, new_x, new_y))

    return min_threshold, None


def IDAStar(start_room, start_x, start_y):
    start_room = [row[:] for row in start_room]

    if start_room[start_x][start_y] == 1:
        start_room[start_x][start_y] = 0

    threshold = heuristic(start_room)

    while True:
        result, res_node = search_with_threshold(start_room, start_x, start_y, threshold)
        if result is True:
            return res_node

        if result == float("inf"):
            return None

        threshold = result


print("Vị trí bắt đầu của máy hút bụi:", (start_x, start_y))
print()

result = IDAStar(room, start_x, start_y)

if result:
    print_solution(result, start_x, start_y)

else:
    print("Không tìm được lời giải")

Nhập ma trận:
Vị trí bắt đầu của máy hút bụi: (1, 0)

Các bước làm sạch phòng:

Bước 0: Vị trí bắt đầu | g(n) = 0 | h(n) = 4 | f(n) = 4
1 1 0 
x 0 0 
1 1 0 

Bước 1: DOWN | g(n) = 1 | h(n) = 3 | f(n) = 4
1 1 0 
0 0 0 
x 1 0 

Bước 2: RIGHT | g(n) = 2 | h(n) = 2 | f(n) = 4
1 1 0 
0 0 0 
0 x 0 

Bước 3: UP | g(n) = 3 | h(n) = 2 | f(n) = 5
1 1 0 
0 x 0 
0 0 0 

Bước 4: UP | g(n) = 4 | h(n) = 1 | f(n) = 5
1 x 0 
0 0 0 
0 0 0 

Bước 5: LEFT | g(n) = 5 | h(n) = 0 | f(n) = 5
x 0 0 
0 0 0 
0 0 0 

Tổng số bước: 5
Cost cuối cùng: 5
